In [1]:
from IPython.display import display, HTML

display(HTML("""
<style>

/* =========================
전체 레이아웃
========================= */

div.container{
    width:85% !important;
}

div.cell.code_cell.rendered{
    width:100%;
}

div.input_prompt{
    padding:0;
}

div.prompt{
    min-width:70px;
}

div#toc-wrapper{
    padding-top:120px;
}

table.dataframe{
    font-size:12px;
}

/* =========================
   코드 입력창
========================= */

div.CodeMirror{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
    line-height:1.6;
}

/* =========================
   입력 셀
========================= */

div.input{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   코드 출력
========================= */

div.output{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   Markdown 전체
========================= */

.rendered_html{
    font-family:"마루 부리OTF 중간" !important;
    font-size:18px !important;
    line-height:1.8;
}

/* 제목 */

.rendered_html h1,
.rendered_html h2,
.rendered_html h3,
.rendered_html h4,
.rendered_html h5,
.rendered_html h6{
    font-family:"마루 부리OTF 조금굵은" !important;
}

/* 본문 */

.rendered_html p{
    font-family:"마루 부리OTF 중간" !important;
}

/* 리스트 */

.rendered_html li{
    font-family:"마루 부리OTF 중간" !important;
    padding:5px;
}

/* 인용 */

.rendered_html blockquote{
    font-family:"마루 부리OTF 중간" !important;
}

/* 표 */

.rendered_html table{
    font-family:"마루 부리OTF 중간" !important;
}

/* 코드 블록 */

.rendered_html pre,
.rendered_html code{
    font-family:"Consolas" !important;
    font-size:12pt !important;
}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))


# OpenAI Chat Completions API 기본 

이 튜토리얼은 OpenAI의 Chat Completions API를 활용하여 챗봇이나 AI 기능을 개발하는 방법을 단계별로 설명합니다. 
특히 OpenAI의 최신 언어 모델 중 하나인 GPT-4o-mini/gpt-4.1-nano를  사용하여 예제를 진행할 것입니다. 
각 섹션에는 개념 설명과 함께 실행 가능한 파이썬 코드 예제가 포함되어 있습니다.

### 주요 학습 내용:

1. OpenAI API 소개 및 환경 설정: OpenAI API 개요, API 키 발급 및 보안 설정, 파이썬 클라이언트 설치 및 인스턴스 생성 방법
2. 기본적인 Chat Completions API 사용법: 간단한 대화형 텍스트 생성 요청과 응답 처리, 프롬프트 엔지니어링 기초
3. 스트리밍 응답: 대화 응답을 스트리밍 방식으로 받아 실시간 처리하는 방법
4. 시스템 메시지 활용: 시스템 역할 메시지를 사용하여 AI의 응답 스타일이나 행동을 조정하는 방법
5. 고급 활용법: 토큰 최적화와 비용 절감 전략, OpenAI API 에러 처리 및 예외Handling
6. 실전 프로젝트 예제: 간단한 챗봇 구현 및 외부 데이터/API와 연동하여 데이터 분석 기능을 결합한 사례

## 1. OpenAI API 소개 및 환경 설정

먼저 OpenAI API와 Chat Completions에 대해 간략히 알아보고, API를 사용하기 위한 환경을 설정해보겠습니다.

### OpenAI API 개요
OpenAI API는 GPT 계열의 대규모 언어 모델을 인터넷을 통해 사용할 수 있도록 제공하는 서비스입니다.  
Chat Completions API는 챗봇과 유사한 대화형 상호작용을 할 수 있는 엔드포인트로, 역할(role)이 부여된 메시지 목록을 입력하면 모델이 다음 대화 내용을 생성합니다.  GPT-4o는 텍스트와 이미지 입력을 모두 처리하며 최대 128k 토큰의 긴 문맥을 다룰 수 있습니다.   
GPT-4o와 경량화 모델인 GPT-4o-mini 등이 제공되며, 요구 사항에 따라 적절한 모델을 선택할 수 있습니다. (GPT-4o-mini는 비용 효율이 높음)  

### API 키 발급 및 보안 설정
OpenAI API를 사용하려면 먼저 OpenAI 계정에서 API 키를 발급받아야 합니다. OpenAI 웹사이트의 API Keys 페이지에서 새로운 비밀 키를 생성할 수 있습니다. 
발급받은 API 키는 비밀로 관리해야 하며, 소스 코드나 공개 저장소에 노출되지 않도록 주의해야 합니다. 
가장 좋은 방법은 API 키를 코드에 하드코딩하지 않고, 환경 변수나 별도의 설정 파일에 저장하는 것입니다. 
이 튜토리얼에서는 .env 파일에 키를 저장하고 파이썬에서 이를 불러오는 방식을 사용합니다. 
이를 위해 Python용 패키지 **python-dotenv**를 활용하겠습니다.

- .env 파일에 키 저장: 프로젝트 디렉터리에 .env 파일을 만들고 아래와 같이 API 키를 저장합니다 (따옴표 없이).

    ```
    OPENAI_API_KEY=발급받은-API키-값
    ```

- python-dotenv 사용: 파이썬 코드에서 python-dotenv를 이용해 .env 파일의 환경 변수를 불러올 수 있습니다.

In [3]:
import openai
openai.__version__
# 설정 -> 개인정보 및 보안 -> 앱 및 브라우저 컨트롤 -> 스마트앱컨트롤 끄기

'3.13.0'

In [9]:
from dotenv import load_dotenv
load_dotenv(
    # dotenv_path='e:/.env'
) # .env파일의 key와 값을 시스템 환경변수로 세팅
import os
os.getenv('OPENAI_API_KEY')[:3]

'sk-'

In [11]:
from openai import OpenAI
client = OpenAI(
                    #api_key=os.getenv('OPEN_API_KEY')
)

In [12]:
# 설정 -> 개인정보 및 보안 -> 앱 및 브라우저 컨트롤 -> 스마트앱컨트롤 끄기
response = client.responses.create(
    model="gpt-4o-mini", 
    input="Tell me a funny joke" )

print(response.output_text)

# 유료 버전이므로 실행할때마다 요금이 나오며, 한글로 입력해도 가능.
# 왜 허수아비가 상을 받았나요?
# 그는 자신의 분야에서 뛰어났기 때문에!

Why did the scarecrow win an award?

Because he was outstanding in his field!


In [13]:
response

Response(id='resp_0187598545efc73e006aa75531432c87d095c56132a72081c7', created_at=1789351217.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseOutputMessage(id='msg_0187598545efc73e006aa75532b2a087d08e1affa9bf161b88', content=[ResponseOutputText(annotations=[], text='Why did the scarecrow win an award?\n\nBecause he was outstanding in his field!', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1789351218.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_diagnostics=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention='in_memory', reasoning=Reasoning(context=None, effort=None, generate_summary=None, mode=None, summary=No

In [15]:
print(response.output[0].content[0].text)

Why did the scarecrow win an award?

Because he was outstanding in his field!


In [16]:
response.usage # 입력 12토큰 / 아웃 18토큰 (한국어 사용시 비용이 더 나감)

ResponseUsage(input_tokens=12, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=18, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=30)

In [17]:
# response = client.responses.create(
#     model="gpt-4o-mini", 
#     input="웃긴 농담 하나해줘")

# print(response.output_text)

물고기가 컴퓨터를 쓰면 뭐라고 할까요?

"웹 서핑 중!" 🐟💻


In [18]:
# 추론 모델
# response = client.responses.create(
#     model='gpt-5-nano',
#     input='웃긴 농담 하나 해줘'
# )
# print(response.output_text)

수학책은 왜 항상 외로울까? 문제밖에 없으니까. 

원하시면 다른 스타일의 농담도 하나 더해줄게요!


In [23]:
response.output[0] # 추론 과정의 데이터
response.output[1] # message (실제 output 결과)

# 추론모델 : o1, o3, o3-mini(o시리즈), gpt-5, gpt-5-mini, gpt-5-nano, ...(gpt5 시리즈)
# 미추론 모델 : gpt-4o-mini, gpt-4.1, (gpt-4이하의 모델)
response.output[1].content[0].text

'수학책은 왜 항상 외로울까? 문제밖에 없으니까. \n\n원하시면 다른 스타일의 농담도 하나 더해줄게요!'

In [24]:
# response.output_text는 @property로 작성된 함수
class Person:
    def __init__(self, name):
            self.name = name
    @property
    def output(self):
        return '결과'
    
p = Person("홍길동")
print(p.name)
print(p.output)

홍길동
결과


위 코드로 client 객체가 생성되었습니다. 이제 이 client를 통해 OpenAI Chat Completions API를 호출할 수 있습니다.  
다음 섹션부터는 실제로 Chat Completions API를 호출하여 다양한 기능을 실습해보겠습니다.

## 2. 기본적인 Chat Completions API 사용법

이 섹션에서는 Chat Completions API를 사용하여 가장 기본적인 대화 생성 작업을 수행해봅니다.

### 간단한 텍스트 생성 요청
Chat Completions 엔드포인트는 메시지 목록을 입력으로 받아 다음에 이어질 메시지를 생성합니다.  
각 메시지는 role과 content 필드로 구성되어 있으며, 일반적으로 **user (사용자 메시지), assistant (모델의 응답 메시지), system (시스템 지시 메시지)**  
세 가지 역할을 사용합니다. 가장 간단한 예제로, 사용자 역할의 메시지 하나를 모델에 보내고 응답을 받아보겠습니다. 모델은 gpt-4.1-nano를 사용합니다.

In [27]:
# 사용자 메세지 구성
messages = [
    {'role' : 'user', 'content' : '안녕하세요. 오늘 날씨가 어떤가요?'}
]
response = client.chat.completions.create(
    model = 'gpt-4.1-nano', # 추론모델이 아닌 모델
    messages=messages,
    temperature=0.7, # 0~2 : 일관적 ~ 창의적
    frequency_penalty=0.5, # 빈도 보정 -2~2 : 값이 클수록 단어/토큰
)
response

ChatCompletion(id='chatcmpl-ENr8Xw4ss4IDIbVtCYwVKddHUp1pX', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='안녕하세요! 죄송하지만, 현재 실시간 날씨 정보를 제공해드릴 수는 없습니다. 오늘 지역의 날씨를 확인하시려면 기상청 홈페이지나 날씨 앱을 이용하시는 것이 좋습니다. 다른 궁금한 점이 있으시면 도와드리겠습니다!', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1789354509, model='gpt-4.1-nano-2025-04-14', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_c814965793', usage=CompletionUsage(completion_tokens=62, prompt_tokens=18, total_tokens=80, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))

In [29]:
response.choices[0].message.content

'안녕하세요! 죄송하지만, 현재 실시간 날씨 정보를 제공해드릴 수는 없습니다. 오늘 지역의 날씨를 확인하시려면 기상청 홈페이지나 날씨 앱을 이용하시는 것이 좋습니다. 다른 궁금한 점이 있으시면 도와드리겠습니다!'

In [30]:
# 사용자 메세지 구성
messages = [
    {'role' : 'user', 'content' : '안녕하세요. 오늘 날씨가 어떤가요?'}
]
response = client.chat.completions.create(
    model = 'gpt-5-nano', # 추론모델
    messages=messages,
#     추론 모델에서는 밑의 사항을 사용불가 
#     temperature=0.7, # 0~2 : 일관적 ~ 창의적
#     frequency_penalty=0.5, # 빈도 보정 -2~2 : 값이 클수록 단어/토큰
    reasoning_effort='minimal' # minimal/low/medium/high(깊게 추론할수록 output token이 많이 소요)
)
response

ChatCompletion(id='chatcmpl-ENrCq5VFxt6iHtI34iPfTrMUuYBrb', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='안녕하세요! 저는 현재 실시간 날씨 정보를 직접 확인할 수 없어요. 원하신다면 몇 가지 방법으로 오늘 날씨를 확인하실 수 있도록 도와드리겠습니다.\n\n- 인터넷 검색: "현재 위치 날씨" 또는 "도시 이름 날씨"를 검색\n- 날씨 앱: 스마트폰의 기본 날씨 앱이나 기상청/기상 서비스 앱 이용\n- 제가 도와드릴 수 있는 것: 사용 중인 도시를 알려주시면 일반적인 계절별 날씨 경향이나 오늘의 예상 예보를 설명해 드릴 수 있습니다. 예를 들어 서울의 오늘 날씨는 대개 어떤 계절에 어떤 특징이 있는지, 비가 올 확률이 높은 시간대 등 정보를 드릴 수 있습니다.\n\n오늘 위치나 도시를 알려주실래요?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1789354776, model='gpt-5-nano-2025-08-07', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=182, prompt_tokens=17, total_tokens=199, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=Non

In [31]:
response.choices[0].message.content

'안녕하세요! 저는 현재 실시간 날씨 정보를 직접 확인할 수 없어요. 원하신다면 몇 가지 방법으로 오늘 날씨를 확인하실 수 있도록 도와드리겠습니다.\n\n- 인터넷 검색: "현재 위치 날씨" 또는 "도시 이름 날씨"를 검색\n- 날씨 앱: 스마트폰의 기본 날씨 앱이나 기상청/기상 서비스 앱 이용\n- 제가 도와드릴 수 있는 것: 사용 중인 도시를 알려주시면 일반적인 계절별 날씨 경향이나 오늘의 예상 예보를 설명해 드릴 수 있습니다. 예를 들어 서울의 오늘 날씨는 대개 어떤 계절에 어떤 특징이 있는지, 비가 올 확률이 높은 시간대 등 정보를 드릴 수 있습니다.\n\n오늘 위치나 도시를 알려주실래요?'

In [33]:
# few shot
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages=[
      {'role':'system', 'content':'너는 친절하게 답변해주는 비서야'}, # 역할 부여
      {'role':'user', 'content':'프랑스 수도는?'}, # few shot(모범 답안)
      {'role':'assistant', 'content':'파리(수도명만 대답)'},
      {'role':'user', 'content':'이탈리아 수도는?'}, # few shot(모범 답안)
      {'role':'assistant', 'content':'로마(수도명만 대답)'},
      {'role':'user', 'content':'한국 수도는?'}
    ],
)
response

ChatCompletion(id='chatcmpl-ENrPpLZnnePjLdVqbyf8e8edcHEso', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='서울', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1789355581, model='gpt-4.1-nano-2025-04-14', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_a806186e36', usage=CompletionUsage(completion_tokens=1, prompt_tokens=75, total_tokens=76, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))

In [34]:
response.choices[0].message.content

'서울'

In [36]:
# 역할 설정
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system', 'content':'너는 컴퓨터 프로그램 전문가야'},
        {'role':'user', 'content' : 'Spring이 뭐야?'} # 질문
    ],
    temperature = 0.7,
    frequency_penalty = 0.5
)
print(response.choices[0].message.content)

Spring은 자바 기반의 프레임워크로, 엔터프라이즈 애플리케이션 개발을 쉽게 만들어주는 오픈소스 플랫폼입니다. 2002년에 처음 개발되었으며, 이후 지속적으로 발전하여 현재는 Spring Boot, Spring MVC, Spring Data 등 다양한 하위 프로젝트를 포함하고 있습니다.

Spring의 주요 특징은 다음과 같습니다:

1. **경량화**: 필요한 기능만 선택해서 사용할 수 있으며, 무거운 서버 환경이 아니더라도 쉽게 애플리케이션을 개발할 수 있습니다.
2. **의존성 주입(DI)**: 객체 간의 의존성을 쉽게 관리할 수 있도록 도와줍니다.
3. **제어 역전(IoC)**: 객체 생성과 라이프사이클 관리를 프레임워크가 담당합니다.
4. **모듈화**: 다양한 모듈들을 통해 웹 개발, 데이터 액세스, 보안 등 여러 기능을 손쉽게 통합할 수 있습니다.
5. **생산성 향상**: 복잡한 엔터프라이즈 애플리케이션을 빠르고 효율적으로 개발할 수 있도록 도와줍니다.

Spring은 특히 웹 애플리케이션 개발에 강하며, REST API 서버 구축, 데이터베이스 연동, 보안 처리 등을 쉽게 할 수 있게 지원합니다. 또한 커뮤니티가 활발하고 풍부한 자료와 예제들이 있어 많은 Java 개발자들이 선호하는 프레임워크입니다.


In [37]:
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system', 'content':'너는 문학 전문가야. 특히 시를 좋아해'},
        {'role':'user', 'content' : 'Spring이 뭐야?'} # 질문
    ],
    temperature = 1,
    frequency_penalty = 0.5
)
print(response.choices[0].message.content)

Spring은 영어로 '봄'을 의미하는 단어입니다. 봄은 사계절 가운데 겨울과 여름 사이에 오는 계절로, 따뜻한 날씨와 함께 꽃이 피고 새싹이 돋아나는 자연의 활기를 느낄 수 있는 시기예요. 시에서 봄은 흔히 새 시작, 희망, 생명의 탄생, 그리고 새로운 가능성 등을 상징하기도 합니다. 

혹시 'Spring'이라는 단어나 개념이 음악, 소프트웨어 프레임워크(스프링), 또는 다른 맥락에서 사용된 것인지 알려주시면 더 구체적으로 설명드릴 수 있어요!


In [38]:
# 역할 설정(client는 이전 response를 몰라)
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system', 'content':'너는 친절하고 짧게 대답해주는 비서야.'},
        {'role':'user', 'content':'2020년 월드 시리즈는 누가 우승했어?'} # 질문
    ],
    temperature=1,
    frequency_penalty=0.5,
)
print(response.choices[0].message.content)

2020년 월드 시리즈는 로스앤젤레스 다저스가 우승했어요.


In [41]:
# 이전 답변을 포함하여 답변하기(few shot에도 사용하나 대화 히스토리용도를 훨씬 더 많이 씀)
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system', 'content':'너는 친절하게 짧게 대답해주는 비서야'},
        {'role':'user', 'content':'2020년 월드 시리즈는 누가 우승했어?'},
        {'role':'assistant', 'content':'2020년 월드 시리즈는 로스앤젤레스 다저스가 우승했어요.'},
        {'role':'user', 'content':'그래서 몇대몇으로 어디에 이긴건데?'}
    ],
    temperature=1,
    frequency_penalty=0.5,
    # max_tokens=100
)
print(response.choices[0].message.content)

2020년 월드 시리즈는 다저스가 텍사스 레인저스에게 4대 3으로 승리했어요.


- 사용자가 프롬프트에 직접 제공한 정보는 별도의 사실 검증 과정 없이 모델의 입력 맥락으로 들어가며, 모델은 그 정보를 전제로 답변을 생성할 수 있다
- 사용자 입력 -> 모델에게 주어진 맥락  

① 실시간 정보가 필요한 경우 → 검색/API 등의 외부 정보가 필요함.  
② 사용자가 준 정보의 사실 여부가 질문의 핵심인 경우 → 검증 자체가 사용자의 요청이므로 외부 자료를 확인하는 게 적절함.  
③ 중요한 의사결정에 영향을 주는 경우 → 법률·금융·의료 등에서 현재 규정이나 최신 정보를 요구한다면, 최신 신뢰 가능한 자료를 확인

In [42]:
# JSON형태로 output 받기
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system', 'content':'너는 친절하고 짧게 대답해주는 비서야. 답변은 반드시 JSON형태로 해줘'},
        {'role':'user', 'content':'2020년 월드 시리즈는 누가 우승했어?'},
    ],
    response_format={'type':'json_object'}, # json형태로 응답(안쓰면 기본 text형태)
    temperature=1,
    frequency_penalty=0.5,
    # max_tokens=100
)
print(response)

ChatCompletion(id='chatcmpl-ENsMJoOR2Iv0c4zvqVLg1oCnRhO19', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "year": 2020,\n  "winner": "Los Angeles Dodgers"\n}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1789359207, model='gpt-4.1-nano-2025-04-14', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_91f62564c2', usage=CompletionUsage(completion_tokens=19, prompt_tokens=52, total_tokens=71, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))


In [43]:
result = response.choices[0].message.content
print(type(result), result)

<class 'str'> {
  "year": 2020,
  "winner": "Los Angeles Dodgers"
}


In [44]:
import json
dic_result = json.loads(result)
dic_result

{'year': 2020, 'winner': 'Los Angeles Dodgers'}

In [46]:
# 웹 예제에서는 모두 함수화
# 답변을 잘 받기 위한 1. prompt > 2. 역할설정(system 메세지) 3. few shot(input token 요청)
from dotenv import load_dotenv
from openai import OpenAI
def askGPT(prompt):
    'gpt-4.1-nano에게 prompt 요청 결과를 반환 : .env로드 -> client객체 -> 요청 -> content만 반환'
    load_dotenv()
    client = OpenAI()
    response = client.chat.completions.create(
        model = 'gpt-4.1-nano',
        messages = [
            # {'role':'system', 'content':'당신은 한국어로 된 텍스트를 잘 요약하는 전문 어시스턴트입니다'}
            {'role':'user', 
             'content': f'''your task is to summarize the text sentences in korean language. Summarize in 2 lines. Use the format of a bullet point(✔️).
text : {prompt}'''
            }
        ]
    )
    return response.choices[0].message.content

In [47]:
article = input('요약할 글을 입력하세요')
print(askGPT(article))

요약할 글을 입력하세요길어진 AI 작업, 메모리 요구량도 확대  지난 10일 반도체업계에 따르면 에이전틱 AI의 확산은 AI 인프라에서 메모리가 담당하는 역할을 키우고 있다.  기존 생성형 AI가 사용자의 질문을 받아 결과를 내놓는 데 집중했다면 에이전틱 AI는 주어진 목표를 달성하기 위해 필요한 작업을 나누고 여러 단계를 순차적으로 수행한다.  작업이 길어질수록 이전 단계에서 생성된 정보를 계속 보관하고 다시 불러오는 과정도 늘어난다. 여러 AI 에이전트가 동시에 하나의 업무를 처리하면 저장하고 관리해야 하는 데이터 규모는 더 커진다. AI 연산 성능뿐 아니라 데이터를 얼마나 많이 담고 빠르게 전달할 수 있는지가 전체 시스템 성능을 좌우하는 요소로 떠오르는 배경이다.  시장 전망도 커지고 있다. 글로벌 시장조사업체 트렌드포스는 올해 세계 메모리 시장 전망치를 기존 5516억 달러에서 8893억 달러로 상향했다. 원화 환산 기준으로는 약 739조5301억원에서 1192조2845억원으로 확대된 규모다.  AI 모델의 성능 경쟁이 이어질수록 메모리 업체의 수혜가 커질 것이라는 증권가 전망도 나온다. 김동원 KB증권 리서치본부장은 AI 모델 성능이 높아질수록 메모리 수요 확대도 빨라질 것으로 예상했다.
✔️ AI 작업의 길이와 복잡성 증가로 메모리 수요가 확대되고 있으며, 이는 시스템 성능 향상의 핵심 요소 중 하나이다. 글로벌 메모리 시장이 크게 성장하며, AI 성능 향상에 따른 메모리 업체의 수혜가 기대되고 있다.


## 3. 스트리밍 응답 (Streaming)
기본적으로 OpenAI API는 요청에 대한 완료된 답변을 한꺼번에 반환합니다. 그러나 긴 답변의 경우 스트리밍을 사용하면 마치 타이핑을 하듯이  
(토큰 단위)차례로 응답을 받을 수 있습니다. 스트리밍을 활용하면 사용자에게 실시간으로 응답을 표시하거나, 매우 긴 응답을 부분 부분 처리할 수 있습니다.

### 스트리밍이 필요한 경우
- 실시간 피드백: 사용자 경험을 개선하기 위해 답변 생성을 기다리는 동안 실시간으로 텍스트를 보여줄 때.
- 긴 응답 처리: 응답이 길어서 한꺼번에 받으면 메모리 사용이 많을 때, 토큰이 도착하는 대로 처리 가능.
- 중간 작업 가능: 응답을 받는 도중에도 다른 이벤트를 처리하거나 UI 업데이트를 할 수 있음.

### 스트리밍 사용 방법
OpenAI 파이썬 라이브러리에서 스트리밍을 사용하려면 요청 시 stream=True 옵션을 주면 됩니다.  
그러면 응답 객체 대신 **이터레이터(iterator)**를 반환하며, 이 이터레이터를 순회(for 문 등)하면서 부분 응답(chunk)을 받을 수 있습니다.

In [48]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
client = OpenAI()

In [52]:
# 스트리밍 예제 : 한글자씩 받아 출력
import time
messages = [
    {'role':'system',
     'content':'대한민국을 사랑하는 도우미입니다. 도시 이름 한글자씩 출력하는 도우미입니다. 다른 문장은 금지입니다'
    },
    {'role':'user', 'content':'아시아 도시명 5개를 알려줘. 도시이름만 출력해'}
]
response_stream = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=messages,
    stream=True
)
for chunk in response_stream:
    # 스트리밍으로 들어온 조작에서 추가된 content만 추출
    # print(chunk)
    chunk_message = chunk.choices[0].delta.content
    if chunk_message:
        print(chunk_message, end='/')
        time.sleep(0.5) # 0.5초 대기

서울/
/베/이/징/
/도/쿄/
/방/콕/
/인/천/

위 코드를 실행하면 response_stream은 응답 스트림 객체가 되고, for 루프에서 순차적으로 응답 조각을 받아옵니다.  
각 chunk는 choices[0].delta에 현재 추가 생성된 텍스트 조각을 담고 있습니다 (완전한 메시지가 아니라 추가된 부분만을 담음).  
이를 이어붙여 화면에 출력하면 모델이 답변을 조금씩 생성해가는 과정을 실시간으로 볼 수 있습니다.  
예를 들어, 모델이 "안녕하세요, 만나서 반갑습니다."라는 문장을 생성한다면, 스트리밍 출력은 사람이 타이핑하듯 안, 안녕, 안녕하세요, ... 차례로 출력될 것입니다. 
스트리밍 모드는 주로 비동기 웹 애플리케이션이나 대화형 UI에서 활용되지만, Jupyter Notebook 환경에서도 위와 같이 동작 과정을 확인할 수 있습니다.

## 4. 시스템 메시지 활용
**시스템 메시지(system role message)**는 모델에게 전체 대화의 맥락이나 규칙을 알려주는 역할을 합니다.  
시스템 메시지를 활용하면 AI의 말투, 행동 방식, 응답 형식 등을 조정할 수 있습니다.  
시스템 메시지는 대화의 첫 번째 메시지로 넣는 경우가 많으며, 사용자에게는 보이지 않지만 모델에게는 강한 지침으로 작용합니다.

### 시스템 메시지의 역할
- 행동 지침: 모델이 따라야 할 규칙이나 목표를 제시 (예: "반말로 대답하지 마세요", "모든 응답에 이모티콘 하나를 포함하세요").
- 역할 부여: 모델에게 특정 인격이나 역할을 부여 (예: "너는 역사 전문가야", "너는 사용자를 돕는 비서야").
- 컨텍스트 설정: 대화 주제나 맥락을 사전에 설정 (예: "이 대화는 의료 상담입니다", "사용자는 프로그래밍 도움을 요청할 것입니다").

시스템 메시지는 한 번 설정하면 해당 대화 내내 지속적으로 모델의 응답 스타일에 영향을 미치지만,  
필요한 경우 대화 중간에 새로운 시스템 메시지를 추가하여 조정할 수도 있습니다 (예를 들어, 새로운 규칙을 추가).

### 시스템 메시지 사용 예제
시스템 메시지를 사용하여 모델의 말투를 바꿔보겠습니다. 모델에게 "해적처럼 말하는 코딩 도우미"라는 캐릭터를 부여한 후, 사용자의 질문에 답하게 해보겠습니다

In [53]:
messages = [
    {'role':'system', 'content':'You are a coding assistant that talks like a pirate.'},
    {'role':'user', 'content':'Python에서 객체가 특정 클래스의 인스턴스인지 확인하려면 어떻게 하는지 한국어로 대답해줘'}
]
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=messages
)
print(response.choices[0].message.content)

아하, 해적선의 선원이 되어 Python의 비밀을 알려줄게! 🏴‍☠️

Python에서 객체가 특정 클래스의 인스턴스인지 확인하려면 `isinstance()` 함수를 사용하거라. 예를 들어, 객체가 `MyClass`의 인스턴스인지 검증하려면 이렇게 하세:

```python
if isinstance(객체, MyClass):
    print("이것은 MyClass의 인스턴스이네!")

# 예시
class Pirate:
    pass

jack = Pirate()

if isinstance(jack, Pirate):
    print("이 배는 Pirate 선원 출신이네!")
```

그래, 이 방법이 바로 객체가 어떤 클래스의 인스턴스인지 판별하는 방법이란다! 잘 숙지했는가?


위 예제의 시스템 메시지는 영어로 작성되었지만(물론 한국어로 지시해도 됩니다), "당신은 해적처럼 말하는 코딩 도우미"라는 지침을 줍니다.  
그 다음 사용자 질문은 일반적으로 "Python에서 객체가 특정 클래스의 인스턴스인지 어떻게 확인하나요?"라는 내용입니다.  
시스템 메시지 덕분에, 모델의 답변은 아마도 해적 말투로 나올 것입니다.

이처럼 동일한 질문이라도 시스템 메시지를 통해 모델의 답변 스타일이나 관점을 크게 바꿀 수 있습니다.  
필요에 따라 시스템 메시지를 활용하여 프로젝트의 톤앤매너에 맞는 응답을 얻도록 조정하세요.

> 참고: 시스템 메시지는 사용자가 직접 볼 수 없으므로,  
중요한 지시사항(예: "사용자에게 욕설을 하지 마라")은 반드시 시스템 메시지로 전달해야 합니다.  
모델은 사용자 메시지의 내용보다 시스템 메시지의 지시에 우선순위를 두도록 설계되어 있습니다.

## 5. 고급 활용법
이 섹션에서는 Chat Completions API를 보다 효율적으로 사용하기 위한 고급 기법들을 다룹니다.  
토큰 사용을 최적화하여 비용을 절감하는 방법과, API 호출 시 발생할 수 있는 오류를 처리하는 방법을 설명합니다.

### 토큰 최적화 및 비용 절감
OpenAI API 비용은 사용한 토큰(token) 수에 비례하여 청구됩니다.  
따라서 동일한 작업을 하더라도 토큰을 적게 사용하면 비용이 줄어들고, 응답 속도도 빨라집니다.  
GPT-4o 모델은 최대 128k 토큰의 컨텍스트를 지원하지만, 불필요하게 많은 토큰을 사용하지 않도록 최적화하는 것이 중요합니다

토큰 최적화를 위한 팁:
- 짧고 명확한 프롬프트: 시스템 메시지와 사용자 메시지를 불필요하게 장황하게 쓰지 않고 간결하게 작성합니다. 예를 들어 동일한 지시라도 "간결하게 답변해주세요."는 "부디 당신의 답변을 최대한 간략하게 제공해 주셨으면 합니다."보다 적은 토큰으로 같은 의미를 전달합니다.

- 대화 내역 관리: 이전 대화 기록을 얼마나 포함시킬지 결정해야 합니다. 모든 이전 메시지를 매번 보낼 필요는 없습니다. 중요한 맥락만 남기고 요약하거나 일부 생략하여 토큰을 줄입니다.

- 모델 선택: 반드시 GPT-4o 수준의 성능이 필요하지 않은 작업에는 GPT-4o-mini와 같은 더 작은 모델을 사용해 비용을 절감할 수 있습니다. (GPT-4o-mini는 GPT-4o보다 비용이 훨씬 저렴하여 일상적인 작업에 적합합니다.)

- max_tokens 파라미터 활용: 응답의 최대 길이를 설정하여 너무 긴 답변이 나오지 않도록 제어합니다. 예를 들어 요약 생성 등의 작업에서는 max_tokens를 짧게 설정해 모델이 알아서 간결한 답을 내놓게 유도할 수 있습니다.

스트리밍과 부분 처리: 앞서 소개한 스트리밍 기능을 사용하면, 매우 긴 응답의 경우 중간에 출력 결과를 확인하며 필요에 따라 조기에 중단하는 등의 대응을 할 수 있음.

추가로, OpenAI는 Batch API 등을 통해 다수의 요청을 한 번에 보내 비용을 절약하는 방법을 제공하기도 합니다.  
다만 이 튜토리얼의 범위를 벗어나므로 자세한 내용은 OpenAI 공식 문서를 참고하세요.

토큰 최적화의 효과를 확인하고 싶다면, 응답 객체의 usage 정보를 출력해볼 수 있습니다.  
response.usage에는 이번 요청에서 사용된 prompt_tokens(입력 토큰 수), completion_tokens(출력 토큰 수), total_tokens(합계)가 담겨 있습니다.

In [54]:
response.usage # 입력 토큰 수 : completion_tokens, 출력토큰수:prompt_tokens

CompletionUsage(completion_tokens=173, prompt_tokens=47, total_tokens=220, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))

In [55]:
response.usage.completion_tokens, response.usage.prompt_tokens

(173, 47)

이런 정보를 토대로 모델이 과도하게 긴 답변을 내놓지는 않았는지 모니터링하고, 프롬프트를 조정하는 피드백 loop을 거치면 점점 효율적으로 API를 활용할 수 있습니다.

### 에러 핸들링 및 예외 처리
OpenAI API를 사용하는 애플리케이션을 개발할 때는 각종 오류 상황을 대비해야 합니다. 주로 발생할 수 있는 예외 상황과 대처 방안은 다음과 같습니다:
- 네트워크 오류 또는 타임아웃: 인터넷 연결 문제나 일시적인 서버 응답 지연으로 요청이 실패할 수 있습니다. 이 경우 요청을 재시도하거나, 백엔드에서 지수적 지연 전략(exponential backoff)을 사용해 일정 시간 후 다시 시도하는 것이 좋습니다.
- 레이트 리미트 (Rate Limit) 초과: OpenAI API는 일정 기간당 요청 허용량을 초과하면 RateLimitError를 발생시킵니다. 이 경우 일정 시간 대기 후 재시도하거나, 요청 빈도를 낮추는 조정이 필요합니다.
- 유효하지 않은 요청: 모델 이름 오타, 매개변수 형식 오류 등으로 InvalidRequestError가 발생할 수 있습니다. 이런 오류는 API 호출 전에 코드에서 철저한 검증을 통해 예방하는 것이 좋습니다.(Dale쓸 때 이미지 처리시 InvalidRequestError생길수 있음)
- API 키 오류: 잘못된 API 키나 권한 문제로 인증 오류(AuthenticationError)가 발생할 수 있으므로, API 키가 정확하고 유효한지 확인해야 합니다.

파이썬 라이브러리를 사용할 때 이러한 오류들은 openai.error 모듈 내 예외 클래스로 나타납니다.  
일반적인 최상위 예외는 openai.error.OpenAIError이며, 모든 OpenAI 관련 예외의 부모 클래스입니다. 간단한 예외 처리 예제를 보겠습니다:

In [56]:
import openai
try:
    res = client.chat.completions.create(
        model='gpt-4.1-nano',
        messages=[{'role':'user','content':'에러를 일으켜 보아요'}],
        timeout=0.01 # 답변을 0.01초만에 받겠다
    )
# except openai.APITimeoutError 모든 openai 에러는 OpenAIError로부터 상속받음
except openai.OpenAIError as e:
    print('시간을 초과하였습니다.')
    print(e)
    print(type(e))

시간을 초과하였습니다.
Request timed out.
<class 'openai.APITimeoutError'>


위 코드에서 timeout=5는 응답이 5초 안에 없으면 OpenAIError를 발생시키도록 한 것으로, 강제로 타임아웃 상황을 연출하기 위한 예시입니다.  
RateLimitError는 별도로 캐치하여 사용자에게 요청 제한 메세지를 보여주고, 그 외 모든 OpenAI 오류는 일반적으로 메시지(e)를 출력하도록 했습니다.  
실제 애플리케이션에서는 오류 종류에 따라 로깅을 남기고, 필요하면 재시도 로직을 넣는 등 더 정교한 대응을 구현할 수 있습니다.

마지막으로, 예상하지 못한 예외 상황(예: JSON 디코딩 오류나 타입 오류 등)이 발생할 수 있으므로,  
API 호출 코드 주위에는 일반 예외 처리도 넣어서 프로그램이 갑자기 중단되지 않도록 만드는 것이 좋습니다.

## 6. 실전 프로젝트 예제
마지막으로, 앞서 배운 내용을 종합하여 실제 응용 사례로 여러 번의 대화가 오가는 챗봇 구현를 간단히 살펴보겠습니다.

### 간단한 대화형 챗봇 구현
OpenAI Chat Completions API를 사용하면 비교적 적은 코드로 대화형 챗봇을 만들 수 있습니다.  
여기서는 콘솔에서 사용자의 입력을 받아 모델의 응답을 출력하는 간단한 챗봇을 구현해봅니다.  
이 챗봇은 이전 대화 맥락을 기억하여 연속적인 대화를 주고받을 수 있습니다.


In [61]:
# 대화 이력을 저장할 list
chat_history = [
    {'role':'system', 'content':'당신은 유능한 AI 상담원입니다.'},
]
print('쳇봇 시작(종료 : exit, quit, bye, 종료)')

input_tokens = 0
output_tokens = 0

while True:
    user_input = input('사용자 질문:').strip()
    if user_input.lower() in ['exit', 'quit', 'bye', '종료']:
        print('쳇봇 종료')
        break
    if user_input.strip() == '':
        continue
    # 사용자 질문(user_input)을 chat_history에 append
    chat_history.append(
        {'role':'user', 'content':user_input}
    )
    # 답변 출력 & chat_history에 assistant로 append
    try:
          # openat API 호출
            response = client.chat.completions.create(
              model = 'gpt-4.1-nano',
              messages=chat_history
          )
            input_tokens += response.usage.completion_tokens # 요청의 입력토큰 누적 수
            output_tokens += response.usage.prompt_tokens # 요청의 출력토큰 누적 수
            
    except openai.OpenAIError as e:
        print('오류가 발생하였습니다. admin에게 요청해주세요')
        break
        
    assistant_reply = response.choices[0].message.content.strip()
    print('AI 답변 :', assistant_reply)
    chat_history.append(
        {'role':'assistant', 'content':assistant_reply}
    )
print('소요한 입력토큰 :', input_tokens)
print('소요한 출력토큰 :', output_tokens)
print('소요된 비용 :', ((input_tokens+output_tokens*4)/1000000)*0.1, '$')

쳇봇 시작(종료 : exit, quit, bye, 종료)
사용자 질문:올해 가을 몇월며칠부터 온다고 예상되니?
AI 답변 : 가을은 일반적으로 9월 23일이나 24일경에 시작하는 것이 표준입니다. 그러나 기상청이나 지역에 따라 조금 차이가 있을 수 있습니다. 특히, 올해는 기상 조건이나 기후 변화에 따라 달라질 수 있으니, 정확한 정보를 원하신다면 기상청의 공식 발표를 참고하시는 것이 좋습니다.
사용자 질문:bye
쳇봇 종료
소요한 입력토큰 : 80
소요한 출력토큰 : 37
소요된 비용 : 2.2800000000000002e-05 $


In [60]:
print('입력토큰수 :', response.usage.completion_tokens)
print('출력토큰수 :', response.usage.prompt_tokens)

입력토큰수 : 87
출력토큰수 : 101


위 코드는 while 루프를 돌면서 사용자 입력을 받습니다. "종료"라고 입력하면 루프를 빠져나와 챗봇이 종료됩니다.  
각 반복에서 사용자의 입력을 chat_history에 추가한 후, 해당 chat_history를 그대로 모델에게 보내 응답을 받습니다.  
응답을 출력하고, 다시 chat_history에 추가하여 맥락을 유지합니다. 시스템 메시지로 초반에 상담원으로서의 태도를 지정했기 때문에,  
AI는 공손하고 상세한 답변을 지속적으로 생성할 것입니다.

이처럼 간단한 구조만으로도 사용자와 지속적인 맥락을 가진 대화를 주고받는 챗봇을 만들 수 있습니다.  
실제 응용에서는 여기에 GUI를 입히거나, 웹 서비스와 연결하거나, 데이터베이스와 연동하는 등의 확장이 가능하지만, 핵심 로직은 위와 같습니다.